In [ ]:
!git clone https://github.com/XPixelGroup/BasicSR.git 2>/dev/null || echo 'BasicSR already cloned'
%cd /content/BasicSR
!pip install -e . -q
!pip install rasterio joblib scikit-learn wandb -q


import sys
sys.path.insert(0, '/content/BasicSR')

/content/BasicSR
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.9/325.9 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 143.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 23.2 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import rasterio

In [ ]:
import wandb
from datetime import datetime

# --- EDIT THESE ---
WANDB_PROJECT = "EMIT-S2-SuperResolution"
WANDB_ENTITY  = None  # your wandb username or team, or None for default
# ------------------

wandb.login()
print(f"WandB ready — project: {WANDB_PROJECT}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: incredet (incredet-ukrainian-catholic-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


WandB ready — project: EMIT-S2-SuperResolution


In [ ]:
!cp "/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/SFIM_npy.zip" /content/
# !mkdir -p /content/data_local && cd /content/data_local && unzip -q /content/SFIM_npy.zip

In [ ]:
!mkdir -p /content/data_local && cd /content/data_local && unzip -q /content/SFIM_npy.zip

In [ ]:
import os, getpass
from google.colab import userdata

os.environ["GH_USER"] = userdata.get("GH_USER")
os.environ["GH_TOKEN"] = userdata.get("GH_TOKEN")
os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")

In [ ]:
import os, textwrap

askpass_path = "/tmp/gh_askpass.sh"
with open(askpass_path, "w") as f:
    f.write(textwrap.dedent("""\
        #!/bin/sh
        case "$1" in
          *Username*) echo "$GH_USER" ;;
          *Password*) echo "$GH_TOKEN" ;;
          *) echo "" ;;
        esac
    """))
os.chmod(askpass_path, 0o700)
os.environ["GIT_ASKPASS"] = askpass_path
os.environ["GIT_TERMINAL_PROMPT"] = "0"

!git clone https://github.com/incredet/HyperSpectralSuperResolution.git

Cloning into 'HyperSpectralSuperResolution'...
remote: Enumerating objects: 360, done.
remote: Counting objects: 100% (21/21), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 360 (delta 8), reused 17 (delta 7), pack-reused 339 (from 1)
Receiving objects: 100% (360/360), 340.96 KiB | 22.73 MiB/s, done.
Resolving deltas: 100% (178/178), done.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio
import re
from tqdm import tqdm

# ── Configuration ──────────────────────────────────────────────
GT_SOURCE   = 'regression'        # 'regression' or 'sfim'
R2_THRESH   = 0.75                # only used when GT_SOURCE == 'regression'
MANIFEST    = Path("/content/r2_manifest.csv")

DATA_ROOT = Path("/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/2026-03-22")
OUT_DIR   = Path("/content/data_local")

EMIT_SCALE      = 10000.0
EMIT_NODATA_U16 = 65535
SCALE           = 6

# ── Discover scene directories ─────────────────────────────────
SCENE_DIRS = {}
for aoi_dir in sorted(DATA_ROOT.glob("aoi_*")):
    for pair_dir in sorted(aoi_dir.iterdir()):
        if not pair_dir.is_dir():
            continue
        key = f"{aoi_dir.name}__{pair_dir.name}"
        SCENE_DIRS[key] = {
            'tiles':      pair_dir / 'tiles',
            'sfim':       pair_dir / 'SFIM_Bilinear',
            'regression': pair_dir / 'regression_synth',
            'aoi':        aoi_dir.name,
        }

# ── AOI-level split ────────────────────────────────────────────
all_aois = sorted(set(v['aoi'] for v in SCENE_DIRS.values()))
rng = np.random.default_rng(42)
perm = rng.permutation(len(all_aois))

n_total = len(all_aois)
N_TEST = max(2, int(0.15 * n_total))
N_VAL  = max(1, int(0.15 * n_total))

TEST_AOIS  = [all_aois[i] for i in perm[:N_TEST]]
VAL_AOIS   = [all_aois[i] for i in perm[N_TEST:N_TEST + N_VAL]]
TRAIN_AOIS = [a for a in all_aois if a not in TEST_AOIS and a not in VAL_AOIS]

print(f"AOIs: {n_total} total  →  {len(TRAIN_AOIS)} train / {len(VAL_AOIS)} val / {len(TEST_AOIS)} test")


# ── R² computation ─────────────────────────────────────────────
def compute_tile_r2(synth_path, emit_b32_path, scale=6):
    """Compare regression_synth (10m) to EMIT-b32 (60m).

    NN-upsamples EMIT to 10m, then computes per-band R².
    Same metric as regression fusion quality.
    """
    with rasterio.open(synth_path) as ds:
        synth = ds.read().astype(np.float32)
    with rasterio.open(emit_b32_path) as ds:
        emit = ds.read().astype(np.float32)

    # NN upsample EMIT to match synth resolution
    emit_up = np.repeat(np.repeat(emit, scale, axis=1), scale, axis=2)

    # Mask nodata and all-zero pixels
    valid = (synth != EMIT_NODATA_U16) & (emit_up != EMIT_NODATA_U16) & (emit_up != 0)
    valid = valid.all(axis=0)

    if valid.sum() < 100:
        return -1.0

    r2_bands = []
    for b in range(synth.shape[0]):
        y_true = emit_up[b][valid]
        y_pred = synth[b][valid]
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - y_true.mean()) ** 2)
        if ss_tot == 0:
            continue
        r2_bands.append(1.0 - ss_res / ss_tot)

    return float(np.mean(r2_bands)) if r2_bands else -1.0


# ── R² manifest (compute once, reuse) ─────────────────────────
from concurrent.futures import ThreadPoolExecutor, as_completed

def build_r2_manifest():
    tasks = []
    for scene_key, paths in tqdm(SCENE_DIRS.items(), desc="Scanning folders"):
        tile_dir = paths['tiles']
        gt_dir   = paths['regression']
        if not gt_dir.exists():
            continue

        # Single glob per directory — scan once
        emit_b32_by_id = {}
        for p in tile_dir.glob("*_emit_b32.tif"):
            m = re.search(r'tile(\d+)', p.stem)
            if m:
                emit_b32_by_id[int(m.group(1))] = p

        synth_by_id = {}
        for p in gt_dir.glob("*_regression_synth.tif"):
            m = re.search(r'tile(\d+)', p.stem)
            if m:
                synth_by_id[int(m.group(1))] = p

        # Match pairs
        for tid in emit_b32_by_id:
            if tid in synth_by_id:
                tasks.append((scene_key, tid, synth_by_id[tid], emit_b32_by_id[tid]))

    print(f"Found {len(tasks)} tile pairs. Computing R²...")
    rows = []

    def _compute(args):
        scene_key, tid, synth_path, emit_path = args
        try:
            r2 = compute_tile_r2(synth_path, emit_path, scale=SCALE)
        except Exception as e:
            r2 = -1.0
        return f"{scene_key},{tid},{r2:.6f}"

    with ThreadPoolExecutor(max_workers=8) as pool:
        futures = {pool.submit(_compute, t): t for t in tasks}
        for f in tqdm(as_completed(futures), total=len(tasks), desc="R²"):
            rows.append(f.result())

    with open(MANIFEST, 'w') as f:
        f.write("scene_key,tile_id,r2_mean\n")
        for row in rows:
            f.write(row + "\n")

    kept = sum(1 for r in rows if float(r.split(',')[2]) >= R2_THRESH)
    print(f"Manifest saved: {MANIFEST}  ({len(rows)} tiles, {kept} with R² ≥ {R2_THRESH})")


def load_r2_manifest():
    r2_map = {}
    with open(MANIFEST) as f:
        next(f)
        for line in f:
            parts = line.strip().split(',')
            scene_key, tid, r2_mean = parts[0], int(parts[1]), float(parts[2])
            r2_map[(scene_key, tid)] = r2_mean
    return r2_map


if GT_SOURCE == 'regression' and not MANIFEST.exists():
    print("Building R² manifest (one-time)...")
    build_r2_manifest()

r2_map = load_r2_manifest() if GT_SOURCE == 'regression' else {}


# ── Read helpers ───────────────────────────────────────────────
def _read_emit_b32(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    mask = arr == EMIT_NODATA_U16
    arr /= EMIT_SCALE
    arr[mask] = 0.0
    return arr


def _read_gt(path):
    with rasterio.open(path) as ds:
        arr = ds.read().astype(np.float32)
    arr /= EMIT_SCALE
    return np.clip(np.nan_to_num(arr, nan=0.0), 0.0, None)


def _save_pair(lr, gt, stem, split):
    assert lr.shape[0] == gt.shape[0], f"Band mismatch: {lr.shape} vs {gt.shape}"
    lr = np.nan_to_num(lr, nan=0.0, posinf=0.0, neginf=0.0)
    gt = np.nan_to_num(gt, nan=0.0, posinf=0.0, neginf=0.0)
    gt = np.clip(gt, 0.0, None)
    np.save(OUT_DIR / split / 'lq' / f"{stem}.npy", lr)
    np.save(OUT_DIR / split / 'gt' / f"{stem}.npy", gt)


# ── Main data preparation ─────────────────────────────────────
def prepare_data():
    for split in ['train', 'val', 'test']:
        (OUT_DIR / split / 'lq').mkdir(parents=True, exist_ok=True)
        (OUT_DIR / split / 'gt').mkdir(parents=True, exist_ok=True)

    counts = {'train': 0, 'val': 0, 'test': 0}
    skipped_r2 = 0
    skipped_missing = 0

    for scene_key, paths in SCENE_DIRS.items():
        aoi = paths['aoi']
        if aoi in TEST_AOIS:
            split = 'test'
        elif aoi in VAL_AOIS:
            split = 'val'
        else:
            split = 'train'

        lr_files = sorted(paths['tiles'].glob("*_emit_b32.tif"))

        if GT_SOURCE == 'sfim':
            gt_dir = paths['sfim']
            gt_pattern = "*_SFIM_fused.tif"
        else:
            gt_dir = paths['regression']
            gt_pattern = "*_regression_synth.tif"

        if not gt_dir.exists():
            continue

        gt_by_id = {}
        for p in gt_dir.glob(gt_pattern):
            m = re.search(r'tile(\d+)', p.stem)
            if m:
                gt_by_id[int(m.group(1))] = p

        for lr_path in lr_files:
            m = re.search(r'tile(\d+)', lr_path.stem)
            if not m:
                continue
            tid = int(m.group(1))

            if tid not in gt_by_id:
                skipped_missing += 1
                continue

            if GT_SOURCE == 'regression':
                r2 = r2_map.get((scene_key, tid), -1.0)
                if r2 < R2_THRESH:
                    skipped_r2 += 1
                    continue

            stem = f"{scene_key}__tile{tid:03d}"
            try:
                lr = _read_emit_b32(lr_path)
                gt = _read_gt(gt_by_id[tid])
                _save_pair(lr, gt, stem, split)
                counts[split] += 1
            except Exception as e:
                print(f"  ERROR {stem}: {e}")

    print(f"\nDone:")
    for s, c in counts.items():
        print(f"  {s}: {c} tiles")
    if GT_SOURCE == 'regression':
        print(f"  Skipped (R² < {R2_THRESH}): {skipped_r2}")
    print(f"  Skipped (missing GT): {skipped_missing}")


prepare_data()

AOIs: 10 total  →  7 train / 1 val / 2 test
Building R² manifest (one-time)...


Scanning folders: 100%|██████████| 10/10 [00:28<00:00,  2.85s/it]


Found 983 tile pairs. Computing R²...


R²: 100%|██████████| 983/983 [07:38<00:00,  2.14it/s]


Manifest saved: /content/r2_manifest.csv  (983 tiles, 673 with R² ≥ 0.75)

Done:
  train: 406 tiles
  val: 118 tiles
  test: 149 tiles
  Skipped (R² < 0.75): 310
  Skipped (missing GT): 24


In [ ]:
if 'PairedNpyDataset' in DATASET_REGISTRY._obj_map:
    del DATASET_REGISTRY._obj_map['PairedNpyDataset']

In [ ]:
import torch
from torch.utils.data import Dataset
from basicsr.utils.registry import DATASET_REGISTRY
from pathlib import Path
import numpy as np
import random


if 'PairedNpyDataset' in DATASET_REGISTRY._obj_map:
    del DATASET_REGISTRY._obj_map['PairedNpyDataset']


@DATASET_REGISTRY.register()
class PairedNpyDataset(Dataset):

    def __init__(self, opt):
        super().__init__()
        self.opt = opt
        self.gt_dir = Path(opt['dataroot_gt'])
        self.lq_dir = Path(opt['dataroot_lq'])
        self.scale = opt.get('scale', 6)
        self.gt_size = opt.get('gt_size', 192)

        self.gt_files = sorted(self.gt_dir.glob('*.npy'))
        self.lq_files = sorted(self.lq_dir.glob('*.npy'))
        assert len(self.gt_files) == len(self.lq_files), \
            f"Mismatch: {len(self.gt_files)} GT vs {len(self.lq_files)} LQ"
        assert len(self.gt_files) > 0, f"No .npy files in {self.gt_dir}"

        for g, l in zip(self.gt_files, self.lq_files):
            assert g.stem == l.stem, f"Name mismatch: {g.stem} vs {l.stem}"

        print(f"[PairedNpyDataset] {len(self.gt_files)} pairs, "
              f"scale={self.scale}, gt_size={self.gt_size}")

    def __len__(self):
        return len(self.gt_files)

    def __getitem__(self, idx):
        gt = np.load(self.gt_files[idx]).astype(np.float32)
        lq = np.load(self.lq_files[idx]).astype(np.float32)

        if self.opt.get('phase') == 'train' and self.gt_size:
            C, H_gt, W_gt = gt.shape
            gt_size = self.gt_size
            lq_size = gt_size // self.scale
            scale = self.scale

            # FIXED: align crop to LR grid boundaries
            max_top_lq  = max(0, H_gt // scale - lq_size)
            max_left_lq = max(0, W_gt // scale - lq_size)
            top_lq  = random.randint(0, max_top_lq)
            left_lq = random.randint(0, max_left_lq)
            top_gt  = top_lq * scale
            left_gt = left_lq * scale

            gt = gt[:, top_gt:top_gt+gt_size, left_gt:left_gt+gt_size]
            lq = lq[:, top_lq:top_lq+lq_size, left_lq:left_lq+lq_size]

            # Augmentation
            if self.opt.get('use_hflip', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=2).copy()
                lq = np.flip(lq, axis=2).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.flip(gt, axis=1).copy()
                lq = np.flip(lq, axis=1).copy()
            if self.opt.get('use_rot', False) and random.random() > 0.5:
                gt = np.rot90(gt, k=1, axes=(1, 2)).copy()
                lq = np.rot90(lq, k=1, axes=(1, 2)).copy()

        gt_tensor = torch.from_numpy(gt.copy()).float()
        lq_tensor = torch.from_numpy(lq.copy()).float()

        return {
            'lq': lq_tensor,
            'gt': gt_tensor,
            'lq_path': str(self.lq_files[idx]),
            'gt_path': str(self.gt_files[idx]),
        }


print("PairedNpyDataset registered.")

PairedNpyDataset registered.


In [ ]:
# import torch.nn as nn
# from basicsr.models.sr_model import SRModel
# from basicsr.utils.registry import MODEL_REGISTRY, LOSS_REGISTRY
# from basicsr.losses import build_loss


# @LOSS_REGISTRY.register()
# class SpectralAngleLoss(nn.Module):
#     def __init__(self, loss_weight=1.0, reduction='mean'):
#         super().__init__()
#         self.loss_weight = loss_weight

#     def forward(self, pred, target, **kwargs):
#         dot = (pred * target).sum(dim=1)
#         norm_pred = pred.norm(dim=1).clamp(min=1e-8)
#         norm_target = target.norm(dim=1).clamp(min=1e-8)
#         cos_sim = (dot / (norm_pred * norm_target)).clamp(-1 + 1e-7, 1 - 1e-7)
#         sam = torch.acos(cos_sim)
#         return self.loss_weight * sam.mean()


# @MODEL_REGISTRY.register()
# class MultiLossSRModel(SRModel):
#     def init_training_settings(self):
#         super().init_training_settings()
#         if self.opt['train'].get('sam_opt'):
#             self.cri_sam = build_loss(self.opt['train']['sam_opt']).to(self.device)
#         else:
#             self.cri_sam = None

#     def optimize_parameters(self, current_iter):
#         self.optimizer_g.zero_grad()
#         self.output = self.net_g(self.lq)

#         l_total = 0
#         loss_dict = {}

#         if self.cri_pix:
#             l_pix = self.cri_pix(self.output, self.gt)
#             l_total += l_pix
#             loss_dict['l_pix'] = l_pix

#         if self.cri_sam:
#             l_sam = self.cri_sam(self.output, self.gt)
#             l_total += l_sam
#             loss_dict['l_sam'] = l_sam

#         l_total.backward()
#         self.optimizer_g.step()

#         self.log_dict = self.reduce_loss_dict(loss_dict)

#         if self.ema_decay > 0:
#             self.model_ema(decay=self.ema_decay)


# print("SpectralAngleLoss + MultiLossSRModel registered.")

In [ ]:
if 'RRDBNet6x' in ARCH_REGISTRY._obj_map:
    del ARCH_REGISTRY._obj_map['RRDBNet6x']

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from basicsr.archs.rrdbnet_arch import RRDB
from basicsr.utils.registry import ARCH_REGISTRY


@ARCH_REGISTRY.register()
class RRDBNet6x(nn.Module):
    """RRDBNet adapted for 6x upsampling.
    - First upsample: 2× nearest-neighbour interpolation + conv
    - Second upsample: 3× nearest-neighbour interpolation + conv
    - Total: 6× spatial upsampling
    """

    def __init__(
        self,
        num_in_ch,
        num_out_ch,
        num_feat=64,
        num_block=23,
        num_grow_ch=32,
    ):
        super().__init__()
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)

        self.body = nn.ModuleList()
        for _ in range(num_block):
            self.body.append(RRDB(num_feat, num_grow_ch=num_grow_ch))
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)

        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)

        nn.init.zeros_(self.conv_last.weight)
        nn.init.zeros_(self.conv_last.bias)

        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

    def forward(self, x):
        feat = self.conv_first(x)

        body_feat = feat
        for block in self.body:
            body_feat = block(body_feat)
        body_feat = self.conv_body(body_feat)
        feat = feat + body_feat

        feat = F.interpolate(feat, scale_factor=2, mode='nearest')
        feat = self.lrelu(self.conv_up1(feat))

        feat = F.interpolate(feat, scale_factor=3, mode='nearest')
        feat = self.lrelu(self.conv_up2(feat))

        out = self.conv_last(self.lrelu(self.conv_hr(feat)))

        base = F.interpolate(x, scale_factor=6, mode='bicubic', align_corners=False)
        return out + base


_net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=64, num_block=4)
_x = torch.randn(1, 32, 16, 16)
_y = _net(_x)
print(f"Input: {_x.shape} → Output: {_y.shape}")
assert _y.shape == (1, 32, 96, 96)
del _net, _x, _y
print("RRDBNet6x registered.")

Input: torch.Size([1, 32, 16, 16]) → Output: torch.Size([1, 32, 96, 96])
RRDBNet6x registered.


In [36]:
# ============================================================
# Cell 5 — Generate training config YAML
# ============================================================

import yaml
from pathlib import Path
from datetime import datetime


# -------- CONFIG — EDIT THESE --------

EXP_NAME = f"Regression-Skip-{datetime.now().strftime('%Y-%m-%d_%H-%M')}"
DATA_ROOT = Path('/content/data_local')
DRIVE_DST  = f'/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/{EXP_NAME}'
NUM_BANDS  = 32
SCALE      = 6
GT_SIZE    = 300 # 192    # crop size in GT domain (192 / 6 = 32 LR pixels)
BATCH_SIZE = 4
TOTAL_ITER = 100_000
LR_RATE    = 2e-4
NUM_BLOCK  = 16     # fewer blocks than default 23 — saves memory, train faster
NUM_FEAT   = 128
# -------------------------------------

config = {
    'name': EXP_NAME,
    'model_type': 'SRModel',
    'scale': SCALE,
    'num_gpu': 1,
    'manual_seed': 42,

    'datasets': {
        'train': {
            'name': 'EMIT_S2_train',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/train/gt',
            'dataroot_lq': f'{DATA_ROOT}/train/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'use_hflip': True,
            'use_rot': True,
            'use_noise': False,
            'use_channel_jitter': False,
            'num_worker_per_gpu': 2,
            'batch_size_per_gpu': BATCH_SIZE,
            'dataset_enlarge_ratio': 100,
        },
        'val': {
            'name': 'EMIT_S2_val',
            'type': 'PairedNpyDataset',
            'dataroot_gt': f'{DATA_ROOT}/val/gt',
            'dataroot_lq': f'{DATA_ROOT}/val/lq',
            'gt_size': GT_SIZE,
            'scale': SCALE,
            'io_backend': {'type': 'disk'},
        },
    },

    'network_g': {
        'type': 'RRDBNet6x',
        'num_in_ch': NUM_BANDS,
        'num_out_ch': NUM_BANDS,
        'num_feat': NUM_FEAT,
        'num_block': NUM_BLOCK,
        'num_grow_ch': 32,
    },

    'path': {
        'pretrain_network_g': None,
        'strict_load_g': True,
        'resume_state': None, # '/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/GLP-32-128/GLP-32-128/training_states/10000.state',
        'experiments_root': DRIVE_DST,
        'models': f'{DRIVE_DST}/models',
        'training_states': f'{DRIVE_DST}/training_states',
        'log': DRIVE_DST,
        'visualization': f'{DRIVE_DST}/visualization',
    },

    'train': {
        'ema_decay': 0.999,
        'optim_g': {
            'type': 'Adam',
            'lr': LR_RATE,
            'weight_decay': 0,
            'betas': [0.9, 0.99],
        },
        'scheduler': {
            'type': 'MultiStepLR',
            'milestones': [15000, 35000, 75000],
            'gamma': 0.5,
        },
        'total_iter': TOTAL_ITER,
        'warmup_iter': -1,
        'grad_clip': {
            'type': 'norm',
            'max_norm': 5.0
        },
        'pixel_opt': {
            'type': 'L1Loss',
            'loss_weight': 1.0,
            'reduction': 'mean',
        },
    },

    'val': {
        'val_freq': 1000,
        'save_img': False,
        'metrics': {
            'psnr': {
                'type': 'calculate_psnr',
                'crop_border': SCALE,
                'test_y_channel': False,
            },
        },
    },

    'logger': {
        'print_freq': 100,
        'save_checkpoint_freq': 10000,
        'use_tb_logger': True,
        'wandb': {
            'project': WANDB_PROJECT,
            'resume_id': None,
        },
    },

    'dist_params': {
        'backend': 'nccl',
        'port': 29500,
    },
}

config_path = Path('/content/BasicSR/options/train') / f'{EXP_NAME}.yml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {config_path}")
print(f"\nKey settings:")
print(f"  Bands: {NUM_BANDS}")
print(f"  Scale: {SCALE}×")
print(f"  GT crop: {GT_SIZE}×{GT_SIZE} → LR crop: {GT_SIZE//SCALE}×{GT_SIZE//SCALE}")
print(f"  Network: RRDBNet6x, {NUM_BLOCK} blocks, {NUM_FEAT} features")
print(f"  Total iterations: {TOTAL_ITER:,}")
print(f"  Batch size: {BATCH_SIZE}")

Config written to: /content/BasicSR/options/train/Regression-Skip-2026-03-24_16-16.yml

Key settings:
  Bands: 32
  Scale: 6×
  GT crop: 300×300 → LR crop: 50×50
  Network: RRDBNet6x, 16 blocks, 128 features
  Total iterations: 100,000
  Batch size: 4


In [ ]:
import importlib
import basicsr.utils.img_util as _iu
import basicsr.models.sr_model as _srm

importlib.reload(_iu)

_orig = _iu.tensor2img

def _patched_tensor2img(tensor, rgb2bgr=True, out_type=np.uint8, min_max=(0, 1)):
    t = tensor[0] if isinstance(tensor, list) else tensor
    if t.ndim >= 3 and t.shape[-3] > 4:
        rgb2bgr = False
    return _orig(tensor, rgb2bgr=rgb2bgr, out_type=out_type, min_max=min_max)

_iu.tensor2img = _patched_tensor2img
_srm.tensor2img = _patched_tensor2img

In [37]:
# ============================================================
# Train
# ============================================================

import sys
sys.path.insert(0, '/content/BasicSR')

from basicsr.train import train_pipeline
import basicsr.train as train_module


# Override sys.argv so BasicSR's argparser sees our config
sys.argv = ['train.py', '-opt', str(config_path)]


# # Launch training (this blocks until done or interrupted)
# import importlib
# importlib.reload(train_module)

# If train_pipeline exists and is callable:
try:
    train_pipeline('/content/BasicSR')
except Exception as e:
    # Fallback: run as subprocess with the registrations injected
    print(f"Direct call failed ({e}), falling back to subprocess...")
    print("NOTE: For subprocess, you need to put the dataset + arch")
    print("      registrations into .py files in BasicSR/basicsr/")
    raise

Disable distributed.


2026-03-24 16:16:15,303 INFO: 
                ____                _       _____  ____
               / __ ) ____ _ _____ (_)_____/ ___/ / __ \
              / __  |/ __ `// ___// // ___/\__ \ / /_/ /
             / /_/ // /_/ /(__  )/ // /__ ___/ // _, _/
            /_____/ \__,_//____//_/ \___//____//_/ |_|
     ______                   __   __                 __      __
    / ____/____   ____   ____/ /  / /   __  __ _____ / /__   / /
   / / __ / __ \ / __ \ / __  /  / /   / / / // ___// //_/  / /
  / /_/ // /_/ // /_/ // /_/ /  / /___/ /_/ // /__ / /<    /_/
  \____/ \____/ \____/ \____/  /_____/\____/ \___//_/|_|  (_)
    
Version Information: 
	BasicSR: 1.4.2
	PyTorch: 2.10.0+cu128
	TorchVision: 0.25.0+cu128
2026-03-24 16:16:15,304 INFO: 
  name: Regression-Skip-2026-03-24_16-16
  model_type: SRModel
  scale: 6
  num_gpu: 1
  manual_seed: 42
  datasets:[
    train:[
      name: EMIT_S2_train
      type: PairedNpyDataset
      dataroot_gt: /content/data_local/train/gt
      dataro

global_step,▁▁▁▂▃▃▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
losses/l_pix,▄▃▃▇▅▇▄▄▇▅▂█▅▆▃▄▅█▇▃▆▂▇█▃▅▂▅▁▄▅▆▅▅▄▇▆▅▃▇
metrics/EMIT_S2_val/psnr,▁▂▅▆▇▇▇▇███████
global_step,15100
losses/l_pix,0.01029
metrics/EMIT_S2_val/psnr,40.0617


2026-03-24 16:16:21,180 INFO: Use wandb logger with id=qvg55pn1; project=EMIT-S2-SuperResolution.
wandb: WARNING When using several event log directories, please call `wandb.tensorboard.patch(root_logdir="...")` before `wandb.init`
2026-03-24 16:16:21,191 INFO: Dataset [PairedNpyDataset] - EMIT_S2_train is built.
2026-03-24 16:16:21,194 INFO: Training statistics:
	Number of train images: 406
	Dataset enlarge ratio: 100
	Batch size per gpu: 4
	World size (gpu number): 1
	Require iter number per epoch: 10150
	Total epochs: 10; iters: 100000.
2026-03-24 16:16:21,197 INFO: Dataset [PairedNpyDataset] - EMIT_S2_val is built.
2026-03-24 16:16:21,198 INFO: Number of val images/folders in EMIT_S2_val: 118


[PairedNpyDataset] 406 pairs, scale=6, gt_size=300
[PairedNpyDataset] 118 pairs, scale=6, gt_size=300


2026-03-24 16:16:21,596 INFO: Network [RRDBNet6x] is created.
2026-03-24 16:16:21,655 INFO: Network: RRDBNet6x, with parameters: 24,564,384
2026-03-24 16:16:21,657 INFO: RRDBNet6x(
  (conv_first): Conv2d(32, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (body): ModuleList(
    (0-15): 16 x RRDB(
      (rdb1): ResidualDenseBlock(
        (conv1): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(160, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv3): Conv2d(192, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv4): Conv2d(224, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv5): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (lrelu): LeakyReLU(negative_slope=0.2, inplace=True)
      )
      (rdb2): ResidualDenseBlock(
        (conv1): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (conv2): Conv2d(160, 32, kernel_size=(3, 

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path

DATA = Path("/content/data_local")
val_lq = sorted((DATA / "val" / "lq").glob("*.npy"))
val_gt = sorted((DATA / "val" / "gt").glob("*.npy"))

psnrs = []
for lq_path, gt_path in zip(val_lq, val_gt):
    lq = np.load(lq_path)
    gt = np.load(gt_path)

    lq_t = torch.from_numpy(lq).unsqueeze(0).float()
    bic = F.interpolate(lq_t, scale_factor=6, mode='bicubic', align_corners=False).squeeze(0).numpy()

    # crop_border=6 to match BasicSR eval
    cb = 6
    bic_crop = bic[:, cb:-cb, cb:-cb]
    gt_crop = gt[:, cb:-cb, cb:-cb]

    mse = np.mean((gt_crop - bic_crop) ** 2)
    if mse > 0:
        psnrs.append(10 * np.log10(1.0 / mse))

print(f"Torch bicubic on val (crop_border=6): {np.mean(psnrs):.2f} +/- {np.std(psnrs):.2f} dB")
print(f"Model is currently at: 39.43 dB")

Torch bicubic on val (crop_border=6): 39.24 +/- 2.69 dB
Model is currently at: 39.43 dB


In [ ]:
DRIVE_DST  = f'/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/{EXP_NAME}'

In [ ]:
import re, csv, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path

# ---- helpers ----

def find_checkpoint(drive_dst, exp_name):
    """Auto-find the latest net_g_*.pth checkpoint. Returns None if nothing found."""
    dirs = [Path(drive_dst)/'models', Path(drive_dst)/exp_name/'models',
            Path('/content/BasicSR/experiments')/exp_name/'models']
    ckpts = []
    for d in dirs:
        if d.exists():
            ckpts.extend(d.glob('net_g_*.pth'))
    if not ckpts:
        print('No checkpoints in:', [str(d) for d in dirs]); return None
    # Filter to only files whose stem contains a number (skip net_g_latest.pth etc.)
    numbered = [(p, int(m.group(1)))
                for p in ckpts
                if (m := re.search(r'(\d+)', p.stem))]
    if not numbered:
        print(f'Found {len(ckpts)} checkpoint(s) but none with numeric names; using last.')
        return ckpts[-1]
    numbered.sort(key=lambda x: x[1])
    print(f'Checkpoint: {numbered[-1][0]}')
    return numbered[-1][0]


def load_model(ckpt_path, num_feat, num_block, device='cuda'):
    net = RRDBNet6x(num_in_ch=32, num_out_ch=32, num_feat=num_feat, num_block=num_block)
    state = torch.load(ckpt_path, map_location='cpu')
    key = 'params_ema' if 'params_ema' in state else 'params' if 'params' in state else None
    net.load_state_dict(state[key] if key else state)
    print(f'Loaded {"EMA" if key=="params_ema" else "regular"} weights')
    return net.to(device).eval()


def to_rgb(cube, bands=(10, 6, 2)):
    rgb = np.stack([cube[b] for b in bands], axis=-1)
    lo, hi = (np.percentile(rgb[rgb > 0], [2, 98]) if (rgb > 0).any() else (0, 1))
    return np.clip((rgb - lo) / (hi - lo + 1e-8), 0, 1)


def compute_psnr(pred, gt, crop_border=0):
    if crop_border > 0:
        pred = pred[:, crop_border:-crop_border, crop_border:-crop_border]
        gt   = gt[:,   crop_border:-crop_border, crop_border:-crop_border]
    mse = np.mean((pred - gt) ** 2)
    return 100.0 if mse < 1e-10 else 10 * np.log10(1.0 / mse)


def compute_sam(pred, gt):
    dot = np.sum(pred * gt, axis=0)
    np_ = np.linalg.norm(pred, axis=0).clip(1e-8)
    ng  = np.linalg.norm(gt,   axis=0).clip(1e-8)
    return np.degrees(np.arccos(np.clip(dot / (np_ * ng), -1 + 1e-7, 1 - 1e-7)).mean())


# ---- visual comparison ----

def plot_visual(data_root, num_feat, num_block,
                ckpt_path=None, drive_dst=None, exp_name=None,
                scale=6, split='val', n_tiles=4, vis_bands=(10, 6, 2),
                device='cuda', save=True):
    """Visual comparison: LR / Bicubic / SR / GT.

    Args:
        ckpt_path:  explicit path to a .pth checkpoint (preferred)
        drive_dst:  experiment dir -- used for auto-search fallback & saving figures
        exp_name:   experiment name -- used with drive_dst for auto-search
    """
    # --- resolve checkpoint ---
    if ckpt_path is not None:
        ckpt = Path(ckpt_path)
        print(f'Checkpoint: {ckpt}')
    else:
        ckpt = find_checkpoint(drive_dst, exp_name)
    if ckpt is None:
        return

    net = load_model(ckpt, num_feat, num_block, device)
    gt_files = sorted(Path(data_root, split, 'gt').glob('*.npy'))
    lq_files = sorted(Path(data_root, split, 'lq').glob('*.npy'))
    if not gt_files:
        print(f'No .npy in {data_root}/{split}/gt'); return
    print(f'{len(gt_files)} tiles in "{split}"')

    idx = np.linspace(0, len(gt_files) - 1, n_tiles, dtype=int)
    fig, axes = plt.subplots(n_tiles, 4, figsize=(20, 5 * n_tiles))
    if n_tiles == 1:
        axes = axes[None, :]
    ps_sr, ps_bic, sm_sr, sm_bic = [], [], [], []

    for row, i in enumerate(idx):
        gt = np.load(gt_files[i]).astype(np.float32)
        lq = np.load(lq_files[i]).astype(np.float32)
        with torch.no_grad():
            sr = net(torch.from_numpy(lq[None]).float().to(device)).squeeze(0).cpu().numpy()
        bic = F.interpolate(torch.from_numpy(lq[None]), scale_factor=scale,
                            mode='bicubic', align_corners=False).squeeze(0).numpy()
        lq_up = F.interpolate(torch.from_numpy(lq[None]), scale_factor=scale,
                              mode='nearest').squeeze(0).numpy()

        p_s = compute_psnr(sr, gt, scale);  p_b = compute_psnr(bic, gt, scale)
        s_s = compute_sam(sr, gt);          s_b = compute_sam(bic, gt)
        ps_sr.append(p_s); ps_bic.append(p_b)
        sm_sr.append(s_s); sm_bic.append(s_b)

        titles = [f'LR (nearest x{scale})',
                  f'Bicubic x{scale}\nPSNR={p_b:.2f}  SAM={s_b:.1f} deg',
                  f'SR\nPSNR={p_s:.2f}  SAM={s_s:.1f} deg',
                  f'GT  {gt.shape[1]}x{gt.shape[2]}']
        for col, (img, t) in enumerate(zip([lq_up, bic, sr, gt], titles)):
            axes[row, col].imshow(to_rgb(img, vis_bands))
            axes[row, col].set_title(t, fontsize=10)
            axes[row, col].axis('off')

    plt.suptitle(f'Visual -- {split} -- bands {list(vis_bands)}', fontsize=14, y=1.01)
    plt.tight_layout()
    if save and drive_dst:
        out = Path(drive_dst) / 'figures'; out.mkdir(parents=True, exist_ok=True)
        fig.savefig(out / f'visual_{split}.png', dpi=150, bbox_inches='tight')
        print(f'Saved -> {out / f"visual_{split}.png"}')
    plt.show()
    print(f'\n--- {split} ({n_tiles} tiles) ---')
    print(f'  SR:      PSNR={np.mean(ps_sr):.2f}+/-{np.std(ps_sr):.2f} dB   SAM={np.mean(sm_sr):.2f}+/-{np.std(sm_sr):.2f} deg')
    print(f'  Bicubic: PSNR={np.mean(ps_bic):.2f}+/-{np.std(ps_bic):.2f} dB   SAM={np.mean(sm_bic):.2f}+/-{np.std(sm_bic):.2f} deg')

plot_visual(
    ckpt_path='/content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/GLP-32-128/GLP-32-128/models/net_g_latest.pth',
    drive_dst=DRIVE_DST,
    data_root='/content/data_local',
    num_feat=NUM_FEAT, num_block=NUM_BLOCK, scale=SCALE,
    split='test', n_tiles=15, vis_bands=(10, 6, 2),
)

Checkpoint: /content/drive/Shareddrives/HyperResData/EMIT_S-2_Matches/GLP-32-128/GLP-32-128/models/net_g_latest.pth


KeyboardInterrupt: 

In [ ]:
import csv, torch, numpy as np, matplotlib.pyplot as plt
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm

def evaluate_full(drive_dst, exp_name, data_root, num_feat, num_block,
                  scale=6, split='test', device='cuda', save=True):
    ckpt = find_checkpoint(drive_dst, exp_name)
    if ckpt is None: return None
    net = load_model(ckpt, num_feat, num_block, device)

    gt_files = sorted(Path(data_root, split, 'gt').glob('*.npy'))
    lq_files = sorted(Path(data_root, split, 'lq').glob('*.npy'))
    assert len(gt_files) == len(lq_files) > 0

    nb = 32
    ps_sr, ps_bic, sm_sr, sm_bic = [], [], [], []
    bm_sr=np.zeros(nb); bm_bic=np.zeros(nb)
    br_sr=np.zeros(nb); br_bic=np.zeros(nb)
    bt=np.zeros(nb); bc=np.zeros(nb)

    for gp, lp in tqdm(zip(gt_files, lq_files), total=len(gt_files), desc=f'Eval {split}'):
        gt = np.load(gp).astype(np.float32)
        lq = np.load(lp).astype(np.float32)
        with torch.no_grad():
            sr = net(torch.from_numpy(lq[None]).float().to(device)).squeeze(0).cpu().numpy()
        bic = F.interpolate(torch.from_numpy(lq[None]), scale_factor=scale,
                            mode='bicubic', align_corners=False).squeeze(0).numpy()
        cb = scale
        sc, bc2, gc = sr[:,cb:-cb,cb:-cb], bic[:,cb:-cb,cb:-cb], gt[:,cb:-cb,cb:-cb]
        ps_sr.append(compute_psnr(sc, gc)); ps_bic.append(compute_psnr(bc2, gc))
        sm_sr.append(compute_sam(sc, gc));  sm_bic.append(compute_sam(bc2, gc))
        for b in range(nb):
            ds=(sc[b]-gc[b]).ravel(); db=(bc2[b]-gc[b]).ravel(); g=gc[b].ravel()
            bm_sr[b]+=np.sum(ds**2); bm_bic[b]+=np.sum(db**2)
            br_sr[b]+=np.sum(ds**2); br_bic[b]+=np.sum(db**2)
            bt[b]+=np.sum((g-g.mean())**2); bc[b]+=len(g)

    R = dict(n=len(gt_files), psnr_sr=ps_sr, psnr_bic=ps_bic, sam_sr=sm_sr, sam_bic=sm_bic,
             band_rmse_sr=np.sqrt(bm_sr/bc), band_rmse_bic=np.sqrt(bm_bic/bc),
             band_psnr_sr=10*np.log10(1/(bm_sr/bc+1e-10)), band_psnr_bic=10*np.log10(1/(bm_bic/bc+1e-10)),
             band_r2_sr=1-br_sr/np.maximum(bt,1e-10), band_r2_bic=1-br_bic/np.maximum(bt,1e-10))

    ps, pb = np.mean(R['psnr_sr']), np.mean(R['psnr_bic'])
    ss, sb = np.mean(R['sam_sr']),  np.mean(R['sam_bic'])
    print(f"\n{'='*60}")
    print(f"  {split.upper()} — {R['n']} tiles")
    print(f"{'='*60}")
    print(f"  {'Metric':<12}{'SR':>12}{'Bicubic':>12}{'Delta':>10}")
    print(f"  {'-'*46}")
    print(f"  {'PSNR (dB)':<12}{ps:>12.4f}{pb:>12.4f}{ps-pb:>+10.4f}")
    print(f"  {'SAM (deg)':<12}{ss:>12.4f}{sb:>12.4f}{ss-sb:>+10.4f}")
    print(f"  {'Mean RMSE':<12}{np.mean(R['band_rmse_sr']):>12.6f}{np.mean(R['band_rmse_bic']):>12.6f}")
    print(f"  {'Mean R2':<12}{np.mean(R['band_r2_sr']):>12.6f}{np.mean(R['band_r2_bic']):>12.6f}")

    # distributions
    fig1, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    a1.hist(R['psnr_sr'], bins=30, alpha=.7, color='#2196F3', label='SR')
    a1.hist(R['psnr_bic'], bins=30, alpha=.7, color='#FF9800', label='Bicubic')
    a1.axvline(ps, color='#1565C0', ls='--', lw=2, label=f'SR={ps:.2f}')
    a1.axvline(pb, color='#E65100', ls='--', lw=2, label=f'Bic={pb:.2f}')
    a1.set(xlabel='PSNR (dB)', ylabel='Count', title='Per-tile PSNR')
    a1.legend(fontsize=8); a1.grid(True, alpha=.3)
    a2.hist(R['sam_sr'], bins=30, alpha=.7, color='#2196F3', label='SR')
    a2.hist(R['sam_bic'], bins=30, alpha=.7, color='#FF9800', label='Bicubic')
    a2.axvline(ss, color='#1565C0', ls='--', lw=2, label=f'SR={ss:.2f} deg')
    a2.axvline(sb, color='#E65100', ls='--', lw=2, label=f'Bic={sb:.2f} deg')
    a2.set(xlabel='SAM (degrees)', ylabel='Count', title='Per-tile SAM')
    a2.legend(fontsize=8); a2.grid(True, alpha=.3)
    plt.suptitle(f'{split} — {R["n"]} tiles', fontsize=13); plt.tight_layout()

    # spectral
    fig2, (b1, b2, b3) = plt.subplots(1, 3, figsize=(18, 5))
    bands = np.arange(nb)
    for ax, k, yl, t in [(b1,'band_psnr','PSNR (dB)','Per-band PSNR'),
                          (b2,'band_rmse','RMSE','Per-band RMSE'),
                          (b3,'band_r2','R2','Per-band R2')]:
        ax.plot(bands, R[f'{k}_bic'], 'o-', ms=4, alpha=.7, color='#FF9800', label='Bicubic')
        ax.plot(bands, R[f'{k}_sr'],  's-', ms=4, alpha=.7, color='#2196F3', label='SR')
        ax.set(xlabel='Band', ylabel=yl, title=t); ax.legend(); ax.grid(True, alpha=.3)
    b3.set_ylim(0, 1.05)
    plt.suptitle(f'Spectral Fidelity — {split}', fontsize=13); plt.tight_layout()

    if save:
        out = Path(drive_dst)/'figures'; out.mkdir(parents=True, exist_ok=True)
        fig1.savefig(out/f'distributions_{split}.png', dpi=150, bbox_inches='tight')
        fig2.savefig(out/f'spectral_{split}.png', dpi=150, bbox_inches='tight')
        with open(out/f'metrics_{split}.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['band','psnr_sr','psnr_bic','rmse_sr','rmse_bic','r2_sr','r2_bic'])
            for b in range(nb):
                w.writerow([b, f"{R['band_psnr_sr'][b]:.4f}", f"{R['band_psnr_bic'][b]:.4f}",
                    f"{R['band_rmse_sr'][b]:.6f}", f"{R['band_rmse_bic'][b]:.6f}",
                    f"{R['band_r2_sr'][b]:.6f}", f"{R['band_r2_bic'][b]:.6f}"])
        with open(out/f'summary_{split}.csv', 'w', newline='') as f:
            w = csv.writer(f)
            w.writerow(['metric','sr_mean','sr_std','bic_mean','bic_std'])
            w.writerow(['PSNR_dB', f'{ps:.4f}', f'{np.std(R["psnr_sr"]):.4f}',
                        f'{pb:.4f}', f'{np.std(R["psnr_bic"]):.4f}'])
            w.writerow(['SAM_deg', f'{ss:.4f}', f'{np.std(R["sam_sr"]):.4f}',
                        f'{sb:.4f}', f'{np.std(R["sam_bic"]):.4f}'])
        print(f'\nExported → {out}/')
    plt.show()
    return R

results = evaluate_full(
    drive_dst=DRIVE_DST, exp_name=EXP_NAME, data_root='/content/data_local/data_local',
    num_feat=NUM_FEAT, num_block=NUM_BLOCK, scale=SCALE, split='test',
)